# Z2005 — Week 8: Hash Tables & Heaps

Programming and Data Structures. This notebook covers how to turn a key into an array index in O(1) average time (hash tables), and how to keep fast access to the single most urgent item as data streams in (heaps and priority queues).


## Learning Objectives

By the end of this notebook you will be able to:

- Write a simple hash function and explain why determinism, uniform distribution, and speed all matter.
- Implement a hash table that resolves collisions by chaining, and trace what happens when two keys collide.
- Implement open addressing with linear probing as a second collision strategy, and explain the trade-off against chaining.
- Explain load factor, why resizing is needed, and connect it to Python's own `dict` implementation.
- Implement a binary min-heap backed by a plain array (`insert`/`heapify_up`, `extract_min`/`heapify_down`), and build one from an unordered array in O(n) with `heapify`.
- Use a heap to build a priority queue, and explain when a heap is the right structure compared to sorting or a balanced BST.


## How to use this notebook

Run the cells top to bottom. Markdown cells explain an idea before any code appears for it, so read them first.

Cells marked `# TODO` are for you to complete — replace `raise NotImplementedError` with working code. Every TODO cell is followed by a self-check cell full of `assert` statements: it prints a friendly message if your code is correct, and raises an `AssertionError` (with no success message) if something is wrong. Fix your code and re-run the TODO cell, then the self-check cell, until it passes.

Full worked solutions are in the very last section, "Solutions" — try each exercise yourself first.


## 1. From key to index: the hash function

A hash table's whole trick is converting an arbitrary key — a string, a tuple, any hashable object — directly into an array index, instead of searching through the array to find where a key lives. This is the coat-check-ticket idea: you hand over a coat, get a numbered ticket, and later the attendant walks straight to that numbered hook rather than checking every coat in the room.

A **hash function** does this conversion. A *good* hash function has three properties: it is **deterministic** (the same key always produces the same index — this is non-negotiable, since otherwise you could never find what you stored), it gives a **uniform distribution** (different keys should spread evenly across all the available slots, not clump together), and it is **fast to compute** (a slow hash function defeats the entire point of avoiding a search).

A common pitfall: a naive hash function that just sums character codes cannot tell anagrams apart — `"cat"` and `"act"` hash to the exact same value, because addition does not care about order. Python's real `hash()` for strings mixes in position information specifically to avoid this.

In [ ]:
def simple_hash(key, table_size: int) -> int:
    """A deliberately simple (and imperfect) hash function.

    Sums the character codes of the key, then reduces to the table's
    range with modulo — the same modulo trick used for circular buffers.
    """
    total = sum(ord(ch) for ch in str(key))  # sum of character codes
    return total % table_size                 # fit into the table's range


# Trace on real keys, by hand, to see the anagram problem directly
print(simple_hash("cat", 10))  # ord('c')=99, ord('a')=97, ord('t')=116 -> 312 % 10 = 2
print(simple_hash("act", 10))  # SAME characters, different order -> 312 % 10 = 2 too!

assert simple_hash("cat", 10) == simple_hash("act", 10)  # the anagram problem, made concrete
print("simple_hash reproduces the anagram collision, as expected")


## 2. Collisions and chaining

A **collision** happens when two different keys hash to the same index. Collisions are not a bug to be engineered away — the pigeonhole principle guarantees them: if there are more possible keys than table slots (which is almost always true), some collision *must* happen eventually, no matter how good the hash function is. The real question a hash table design has to answer is never "can we avoid collisions" but "how do we handle them well."

**Chaining** is the simplest answer: each array slot ("bucket") holds a *list* of key-value pairs rather than a single value. Colliding keys simply share a bucket, appended to its list. Retrieval hashes to the right bucket in O(1), then scans that bucket's (hopefully short) list to find the matching key — that scan is the only extra cost collisions introduce.

In [ ]:
class HashTable:
    def __init__(self, size: int = 10):
        self.size = size
        self.buckets = [[] for _ in range(size)]  # each slot holds a LIST (a "chain")

    def _hash(self, key) -> int:
        return sum(ord(ch) for ch in str(key)) % self.size

    def put(self, key, value) -> None:
        index = self._hash(key)
        bucket = self.buckets[index]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)  # key already present: update in place
                return
        bucket.append((key, value))       # new key: append to the chain

    def get(self, key):
        index = self._hash(key)
        for k, v in self.buckets[index]:
            if k == key:
                return v
        raise KeyError(key)               # bucket scanned fully, key not found

    def delete(self, key) -> None:
        index = self._hash(key)
        bucket = self.buckets[index]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                del bucket[i]
                return
        raise KeyError(key)


# Trace with a deliberate collision: "dog" and "bat" both hash to index 4 when size=5
ht = HashTable(size=5)
ht.put("cat", 1)
ht.put("dog", 2)
ht.put("bat", 3)   # collides with "dog" at the same bucket -> chained, not overwritten
assert ht.get("bat") == 3
assert ht.get("dog") == 2
ht.delete("dog")
assert ht.get("bat") == 3   # "bat" survives "dog" being removed from the same bucket
print("Chaining hash table checks passed")


## 3. Open addressing and linear probing

Chaining is not the only way to resolve collisions. **Open addressing** keeps every entry directly in the array itself — no separate lists at all. On a collision, it *probes* forward to find the next free slot. **Linear probing** is the simplest probing rule: try `index + 1`, then `index + 2`, and so on (wrapping around with modulo), until an empty slot is found.

The trade-off against chaining is real in both directions: open addressing has better memory locality (no separate list allocations scattered across memory, which matters for cache performance), but it is more sensitive to a high load factor — as the table fills up, probe sequences get longer — and deletion is delicate, since naively clearing a slot can break the probe sequence for keys that were placed *after* it. (Production implementations use a special "deleted" marker rather than truly emptying the slot, for exactly this reason.)

In [ ]:
class OpenAddressingHashTable:
    def __init__(self, size: int = 10):
        self.size = size
        self.slots = [None] * size  # each slot holds a (key, value) pair or None

    def _hash(self, key) -> int:
        return sum(ord(ch) for ch in str(key)) % self.size

    def put(self, key, value) -> None:
        index = self._hash(key)
        while self.slots[index] is not None and self.slots[index][0] != key:
            index = (index + 1) % self.size  # linear probing: try the NEXT slot
        self.slots[index] = (key, value)

    def get(self, key):
        index = self._hash(key)
        start = index
        while self.slots[index] is not None:
            if self.slots[index][0] == key:
                return self.slots[index][1]
            index = (index + 1) % self.size
            if index == start:   # travelled the whole table without finding it
                break
        raise KeyError(key)


oa = OpenAddressingHashTable(size=5)
oa.put("cat", 1)   # hashes to some index i
oa.put("dog", 2)
oa.put("bat", 3)   # if this collides with "dog", it probes forward to the next free slot
assert oa.get("cat") == 1
assert oa.get("dog") == 2
assert oa.get("bat") == 3
print("Open addressing checks passed — all three keys land in distinct slots via probing")


## 4. Load factor, resizing, and Python's own `dict`

The **load factor** \(\alpha = n / \text{size}\) is the average number of entries per slot. With a good hash function and \(\alpha\) kept small (typically well under 1), the average chain length stays O(1), so the average-case cost of `put`/`get`/`delete` is O(1) hash computation plus O(1) average scan = **O(1) average**. If \(\alpha\) grows too large, chains (or probe sequences) get long, and performance degrades — in the worst case, if every key collided into one bucket, every operation would cost O(n), no better than a plain linked list.

The fix is the same idea as a dynamic array from earlier in the course: **resize**. When the load factor crosses a threshold, allocate a bigger backing array and re-insert every existing entry into it (this re-insertion is necessary because indices depend on `size`, which just changed). Resizing is an expensive O(n) operation, but it happens rarely enough that the *amortized* cost per insertion stays O(1) — exactly the argument used for dynamic array growth.

Python's built-in `dict` is a real, highly-optimized hash table using **open addressing** (not chaining), tuned with a specific probing scheme and automatic resizing. Since Python 3.7, `dict` also preserves insertion order as an implementation detail — but the O(1) average lookup guarantee is exactly the hash table property built from scratch above.

In [ ]:
import timeit

def build_chaining_table(n: int) -> HashTable:
    table = HashTable(size=n)  # low load factor: size == n, so alpha stays around 1 at most
    for i in range(n):
        table.put(f"key{i}", i)
    return table

def build_python_dict(n: int) -> dict:
    d = {}
    for i in range(n):
        d[f"key{i}"] = i
    return d

n = 2000
chaining_table = build_chaining_table(n)
python_dict = build_python_dict(n)

# Live timing: look up the same key 1000 times in each structure
chain_time = timeit.timeit(lambda: chaining_table.get("key1999"), number=1000)
dict_time = timeit.timeit(lambda: python_dict["key1999"], number=1000)

print(f"HashTable (chaining, our implementation): {chain_time:.6f}s for 1000 lookups")
print(f"dict (Python's open-addressing C implementation): {dict_time:.6f}s for 1000 lookups")
print("Both are O(1) average — the gap here is implementation overhead (Python loop vs. C), not algorithmic complexity")


## 5. Heaps: fast access to "most urgent," not full ordering

Think of a hospital emergency room: patients are not treated in arrival order (that is a queue), and they are not kept in a fully sorted line either — re-sorting the whole line every time a more urgent patient walks in would be wasteful. What is actually needed is fast access to "who is most urgent right now," plus fast insertion of a new patient at the correct position. Nothing more.

A **heap** provides exactly this, with a much weaker guarantee than a sorted structure. The **heap property** for a min-heap is only *local*: every parent is less than or equal to both of its children. Compare this to a balanced binary search tree, which guarantees *full* ordering everywhere (left subtree < node < right subtree). A heap's narrower guarantee is precisely what makes it cheap: it enables O(1) access to the minimum and O(log n) insert/extract, but — unlike a BST — it gives no help at all for finding an arbitrary value; that still costs O(n), no better than an unsorted array.

A heap is always a **complete tree**: every level is full except possibly the last, which fills left to right with no gaps. Completeness is what lets a heap be stored in a plain array with no pointers at all — parent/child relationships are computed by index arithmetic, not by following references.

In [ ]:
class MinHeap:
    def __init__(self):
        self.data = []  # a PLAIN array -- no TreeNode, no left/right pointers

    def _parent(self, i: int) -> int:
        return (i - 1) // 2

    def _left(self, i: int) -> int:
        return 2 * i + 1

    def _right(self, i: int) -> int:
        return 2 * i + 2

    def peek(self):
        return self.data[0]  # the minimum is always at the root, O(1)


# Trace the index formulas on a concrete array, by hand
h = MinHeap()
h.data = [1, 3, 2, 7, 4, 5]
#          0  1  2  3  4  5   <- indices

assert h._parent(3) == 1     # arr[1]=3 is the parent of arr[3]=7
assert h._left(1) == 3       # arr[3]=7 is the left child of arr[1]=3
assert h._right(1) == 4      # arr[4]=4 is the right child of arr[1]=3
assert h._parent(0) == -1    # the root has no real parent -- index -1 signals that
print("Array-index parent/child formulas check out")


## 6. Insert (`heapify_up`) and extract-min (`heapify_down`)

**Insert** always places the new value at the array's end — the next open leaf position, which automatically preserves completeness. If the new value violates the heap property relative to its parent (i.e. it is smaller, for a min-heap), swap it upward and repeat, until either the property holds or the root is reached. This "bubble upward" is called `heapify_up`, and since a complete tree of \(n\) nodes always has height O(log n), the whole operation costs O(log n) — this height guarantee is exactly what a plain, unbalanced BST cannot promise.

**Extract-min** removes and returns the root (the minimum). To keep the tree complete, move the very *last* element into the now-empty root position, then bubble it downward (`heapify_down`): at each step, compare against both children and swap with the smaller one, repeating until the heap property holds again. This also costs O(log n).

A common pitfall: forgetting to compare against *both* children in `heapify_down` and picking the wrong one to swap with breaks the heap property silently — always swap with the smaller of the two children, not just the left one.

In [ ]:
class MinHeap(MinHeap):  # extend the class above with insert/extract, same object
    def insert(self, value) -> None:
        self.data.append(value)                       # add at the very end (next open leaf)
        self._heapify_up(len(self.data) - 1)

    def _heapify_up(self, i: int) -> None:
        parent = self._parent(i)
        if i > 0 and self.data[i] < self.data[parent]:
            self.data[i], self.data[parent] = self.data[parent], self.data[i]
            self._heapify_up(parent)                   # keep checking further up

    def extract_min(self):
        min_value = self.data[0]
        last = self.data.pop()                          # remove the LAST element
        if self.data:
            self.data[0] = last                          # move it to the root
            self._heapify_down(0)
        return min_value

    def _heapify_down(self, i: int) -> None:
        smallest = i
        for child in (self._left(i), self._right(i)):
            if child < len(self.data) and self.data[child] < self.data[smallest]:
                smallest = child                          # track smaller of the two children
        if smallest != i:
            self.data[i], self.data[smallest] = self.data[smallest], self.data[i]
            self._heapify_down(smallest)                  # keep bubbling down


# Trace insert(0) on [1, 3, 2, 7, 4, 5]
h = MinHeap()
for v in [1, 3, 2, 7, 4, 5]:
    h.insert(v)
h.insert(0)
assert h.data == [0, 3, 1, 7, 4, 5, 2]   # 0 bubbles all the way to the root

# Trace extract_min on that same heap
assert h.extract_min() == 0
assert h.data == [1, 3, 2, 7, 4, 5]      # last element (2) moved to root, then bubbled down
print("insert/heapify_up and extract_min/heapify_down checks passed")


## 7. Building a heap in O(n), and heap sort

Inserting \(n\) values one at a time costs O(n log n) overall. But if you already have the *whole* array up front, there is a faster way to arrange it into a valid heap: starting from the last non-leaf node and working backward to the root, run `heapify_down` at each position. Every leaf is trivially already a valid (single-node) heap, so this only does real work on internal nodes — and a careful accounting of the work at each level (most nodes are near the leaves, where `heapify_down` has little distance to travel) shows the total cost is O(n), not O(n log n). This `heapify` function is what Python's own `heapq.heapify` implements.

**Heap sort** falls straight out of the heap operations already built: `heapify` the array in O(n), then call `extract_min` repeatedly — each `extract_min` costs O(log n), and there are n of them, giving O(n log n) overall, matching the best comparison-based sorts from earlier in the course.

In [ ]:
def _heapify_down_arr(arr: list, n: int, i: int) -> None:
    smallest = i
    left, right = 2 * i + 1, 2 * i + 2
    if left < n and arr[left] < arr[smallest]:
        smallest = left
    if right < n and arr[right] < arr[smallest]:
        smallest = right
    if smallest != i:
        arr[i], arr[smallest] = arr[smallest], arr[i]
        _heapify_down_arr(arr, n, smallest)

def heapify(arr: list) -> None:
    """Rearrange arr into a valid min-heap, in place, in O(n)."""
    n = len(arr)
    for i in range(n // 2 - 1, -1, -1):  # start at the last non-leaf, work back to the root
        _heapify_down_arr(arr, n, i)

def heap_sort(arr: list) -> list:
    heap = MinHeap()
    heap.data = arr[:]      # copy, so the caller's list is untouched
    heapify(heap.data)      # O(n) build
    return [heap.extract_min() for _ in range(len(arr))]  # n extractions, O(log n) each


assert heap_sort([5, 2, 8, 1, 9, 3]) == [1, 2, 3, 5, 8, 9]

import random
random_data = [random.randint(0, 1000) for _ in range(200)]
assert heap_sort(random_data) == sorted(random_data)
print("heapify (O(n) build) and heap_sort checks passed")


## 8. Priority queues

A **priority queue** is a *disciplined interface* over a heap, in exactly the same spirit as `Stack` and `Queue` earlier in the course being disciplined interfaces over a plain array: it hides the array-index arithmetic behind names like `push`/`pop`, which prevents misuse and makes the intent of the code obvious to a reader.

Storing `(priority, item)` tuples and letting Python's own tuple comparison do the ordering is a common, elegant pattern: tuples compare element by element, so the tuple with the smaller `priority` is always considered smaller overall, no matter what `item` is (Python only inspects `item` to break a tie if two priorities are exactly equal).

Priority queues show up whenever a system needs to repeatedly ask for "the single most urgent thing right now" as data continuously arrives — a task scheduler, an event simulation, or Dijkstra's shortest-path algorithm's core loop are all exactly this pattern. This is a genuinely different problem from *sorting* (arrange everything, once) or *searching* (look up a specific key, continuously) — using a full sort to repeatedly read just the top few items, re-sorting from scratch on every call, is a common and costly design mistake where a small heap does the same job far more cheaply.

In [ ]:
class PriorityQueue:
    def __init__(self):
        self._heap = MinHeap()

    def push(self, priority, item) -> None:
        self._heap.insert((priority, item))  # tuples compare element-wise: priority first

    def pop(self):
        priority, item = self._heap.extract_min()
        return item

    def is_empty(self) -> bool:
        return len(self._heap.data) == 0


scheduler = PriorityQueue()
scheduler.push(3, "send weekly report")
scheduler.push(1, "handle server outage")
scheduler.push(2, "review pull request")

order = []
while not scheduler.is_empty():
    order.append(scheduler.pop())

assert order == ["handle server outage", "review pull request", "send weekly report"]
print("PriorityQueue processed tasks in priority order:", order)


## Exercises

Try each of these yourself before looking at the Solutions section at the end.

### Exercise 1 — `contains` for the chaining hash table

Add a `contains(key)` method to a hash table (chaining-based) that returns `True` if the key is present and `False` otherwise, *without* raising an exception.

Example: for a table containing `"cat"` and `"dog"`, `contains("cat")` returns `True` and `contains("fish")` returns `False`.


In [ ]:
class HashTableWithContains(HashTable):
    def contains(self, key) -> bool:
        """Return True if key is present in the table, False otherwise.

        Must not raise KeyError -- unlike get(), a missing key is a normal,
        expected outcome here, not an error.
        """
        # TODO: implement this
        raise NotImplementedError


### Exercise 2 — load factor

Write a function `load_factor(table)` that computes \(\alpha = n / \text{size}\) for a `HashTable`, where `n` is the total number of stored key-value pairs (summed across all buckets) and `size` is `table.size`.

Example: a table of `size=5` holding 3 entries total (across all buckets) has `load_factor` `0.6`.


In [ ]:
def load_factor(table: "HashTable") -> float:
    """Return n / size for the given HashTable, where n is the total
    number of stored key-value pairs across all buckets.
    """
    # TODO: implement this
    raise NotImplementedError


### Exercise 3 — `is_valid_min_heap`

Write a function `is_valid_min_heap(arr)` that checks whether a plain Python list satisfies the min-heap property (every parent less than or equal to both of its children), returning `True` or `False`. Use the same array-index parent/child arithmetic from Section 5 — do not build a `MinHeap` object.

Example: `is_valid_min_heap([1, 3, 2, 7, 4, 5])` is `True`. `is_valid_min_heap([5, 3, 2])` is `False`, because `5 > 3`.


In [ ]:
def is_valid_min_heap(arr: list) -> bool:
    """Return True if arr satisfies the min-heap property at every index,
    False otherwise. Use index arithmetic directly -- do not construct a
    MinHeap object for this.
    """
    # TODO: implement this
    raise NotImplementedError


### Exercise 4 — `kth_smallest` using a heap

Write a function `kth_smallest(arr, k)` that returns the k-th smallest value in `arr` (1-indexed: `k=1` means the smallest) using a `MinHeap` — build a heap from `arr` with `heapify`, then call `extract_min` exactly `k` times.

Example: `kth_smallest([5, 2, 8, 1, 9, 3], 3)` returns `3`, because the sorted order is `[1, 2, 3, 5, 8, 9]` and the 3rd smallest is `3`.


In [ ]:
def kth_smallest(arr: list, k: int):
    """Return the k-th smallest value in arr (1-indexed), using a heap.

    Must run in O(n + k log n): build the heap once in O(n) with heapify,
    then extract_min k times.
    """
    # TODO: implement this
    raise NotImplementedError


### Exercise 5 (harder) — `merge_k_sorted`

Write a function `merge_k_sorted(lists)` that merges `k` already-sorted lists of integers into one sorted list, using a `PriorityQueue`. Push each list's first element with its value as priority (tagged with which list and index it came from); each `pop` gives the next value in overall sorted order, after which push that list's next element, if any.

Example: `merge_k_sorted([[1, 4, 7], [2, 5], [3, 6, 9]])` returns `[1, 2, 3, 4, 5, 6, 7, 9]`.


In [ ]:
def merge_k_sorted(lists: list) -> list:
    """Merge k already-sorted lists of integers into one sorted list,
    using a PriorityQueue (not Python's built-in sorted()/heapq).
    """
    # TODO: implement this
    raise NotImplementedError


## Self-Check

Run each check after implementing the matching exercise above.

In [ ]:
# Self-check: Exercise 1
table = HashTableWithContains(size=5)
table.put("cat", 1)
table.put("dog", 2)
assert table.contains("cat") is True
assert table.contains("dog") is True
assert table.contains("fish") is False
table.delete("cat")
assert table.contains("cat") is False
print("Exercise 1 passed")


In [ ]:
# Self-check: Exercise 2
t = HashTable(size=5)
t.put("a", 1)
t.put("b", 2)
t.put("c", 3)
assert load_factor(t) == 0.6
empty = HashTable(size=4)
assert load_factor(empty) == 0.0
print("Exercise 2 passed")


In [ ]:
# Self-check: Exercise 3
assert is_valid_min_heap([1, 3, 2, 7, 4, 5]) is True
assert is_valid_min_heap([0, 3, 1, 7, 4, 5, 2]) is True
assert is_valid_min_heap([5, 3, 2]) is False
assert is_valid_min_heap([]) is True     # an empty array is trivially a valid heap
assert is_valid_min_heap([42]) is True   # a single node is trivially a valid heap
print("Exercise 3 passed")


In [ ]:
# Self-check: Exercise 4
assert kth_smallest([5, 2, 8, 1, 9, 3], 1) == 1
assert kth_smallest([5, 2, 8, 1, 9, 3], 3) == 3
assert kth_smallest([5, 2, 8, 1, 9, 3], 6) == 9
print("Exercise 4 passed")


In [ ]:
# Self-check: Exercise 5
result = merge_k_sorted([[1, 4, 7], [2, 5], [3, 6, 9]])
assert result == [1, 2, 3, 4, 5, 6, 7, 9]
assert merge_k_sorted([[1, 2, 3]]) == [1, 2, 3]
assert merge_k_sorted([[], [1, 2]]) == [1, 2]
print("Exercise 5 passed")


## Quiz

**1. Why is a hash table's average-case lookup O(1) but its worst-case lookup O(n)?**

<details><summary>Show answer</summary>
Average case assumes a good, uniform hash function with a low load factor, so each bucket holds roughly O(1) entries and the scan after hashing is short. Worst case happens if every key hashes to the same bucket (a bad hash function, or an adversary deliberately choosing colliding keys) — then the whole table degenerates into one long chain, and a lookup has to scan all n entries, exactly like an unsorted linked list.
</details>

**2. A heap gives O(1) access to the minimum. Why can't it also give fast access to an arbitrary value, the way a balanced BST can?**

<details><summary>Show answer</summary>
The heap property only guarantees that a parent is less than or equal to its children — it says nothing about the relative order of left and right subtrees, or about any node compared to a non-ancestor/non-descendant. A BST's stronger left-less-than-right-at-every-node guarantee is what lets binary search rule out half the remaining tree at each step. A heap's narrower guarantee means finding an arbitrary value may require checking every node, O(n), same as an unsorted array — this is the deliberate trade for O(1) minimum access.
</details>

**3. Why is building a heap from an unordered array with `heapify` O(n), when inserting the same n values one at a time is O(n log n)?**

<details><summary>Show answer</summary>
Inserting one at a time calls heapify_up n times, and each call can travel up to O(log n) levels in the worst case, giving O(n log n) overall. Bottom-up heapify instead calls heapify_down starting from the last non-leaf node up to the root. Most nodes are near the bottom of the tree, where heapify_down has very little distance left to travel (leaves need none at all), so the total work summed across all levels comes out to O(n), not O(n log n) — a smaller amount of work is done more often, rather than a larger amount of work being done at every node.
</details>

**4. A dashboard needs to show the 3 most urgent items out of thousands, refreshed constantly as new items arrive. Why is repeatedly sorting the entire list and taking the first 3 the wrong structure, and what should be used instead?**

<details><summary>Show answer</summary>
Sorting the full list every refresh costs O(n log n) just to read 3 values — most of that ordering work (positions 4 through n) is thrown away immediately. This is a sorting solution applied to a priority-processing problem. The right structure is a small heap (or a priority queue built on one): maintaining it incrementally as items arrive costs O(log n) per update, and reading the top items is O(1) or O(k log n) for k items, with no wasted full-ordering work.
</details>


## Solutions (try the exercises yourself first!)

In [ ]:
# Solution: Exercise 1
class HashTableWithContainsSolution(HashTable):
    def contains(self, key) -> bool:
        index = self._hash(key)
        for k, v in self.buckets[index]:
            if k == key:
                return True
        return False

t = HashTableWithContainsSolution(size=5)
t.put("cat", 1)
assert t.contains("cat") is True
assert t.contains("fish") is False
print("Exercise 1 solution verified")


In [ ]:
# Solution: Exercise 2
def load_factor_solution(table: "HashTable") -> float:
    n = sum(len(bucket) for bucket in table.buckets)
    return n / table.size

t = HashTable(size=5)
t.put("a", 1)
t.put("b", 2)
t.put("c", 3)
assert load_factor_solution(t) == 0.6
print("Exercise 2 solution verified")


In [ ]:
# Solution: Exercise 3
def is_valid_min_heap_solution(arr: list) -> bool:
    n = len(arr)
    for i in range(n):
        left, right = 2 * i + 1, 2 * i + 2
        if left < n and arr[i] > arr[left]:
            return False
        if right < n and arr[i] > arr[right]:
            return False
    return True

assert is_valid_min_heap_solution([1, 3, 2, 7, 4, 5]) is True
assert is_valid_min_heap_solution([5, 3, 2]) is False
print("Exercise 3 solution verified")


In [ ]:
# Solution: Exercise 4
def kth_smallest_solution(arr: list, k: int):
    heap = MinHeap()
    heap.data = arr[:]
    heapify(heap.data)
    value = None
    for _ in range(k):
        value = heap.extract_min()
    return value

assert kth_smallest_solution([5, 2, 8, 1, 9, 3], 3) == 3
print("Exercise 4 solution verified")


In [ ]:
# Solution: Exercise 5
def merge_k_sorted_solution(lists: list) -> list:
    pq = PriorityQueue()
    for i, lst in enumerate(lists):
        if lst:
            pq.push(lst[0], (i, 0))  # (list_index, element_index)
    result = []
    while not pq.is_empty():
        i, idx = pq.pop()
        value = lists[i][idx]
        result.append(value)
        if idx + 1 < len(lists[i]):
            pq.push(lists[i][idx + 1], (i, idx + 1))
    return result

assert merge_k_sorted_solution([[1, 4, 7], [2, 5], [3, 6, 9]]) == [1, 2, 3, 4, 5, 6, 7, 9]
print("Exercise 5 solution verified")


## MTech Extension — open addressing with tombstones, and why deletion breaks naive probing

Open addressing's deletion problem, mentioned briefly in Section 3, deserves a careful look, because the naive fix is a classic source of silent bugs.

Suppose three keys probe into a chain: `A` at index 2, then `B` collides and lands at index 3, then `C` also collides and lands at index 4. If `B` is deleted by simply setting slot 3 back to `None`, a later `get("C")` starts at `C`'s home index 2, finds it occupied by `A`, probes to 3 — and now finds `None`. A naive open-addressing `get` stops probing as soon as it sees an empty slot, on the reasonable assumption that an empty slot means "nothing was ever inserted past this point." That assumption is now false: `C` is still in the table, sitting at index 4, unreachable.

The standard fix is a **tombstone**: a special marker distinct from both "empty" and "occupied," left behind by deletion. `get` keeps probing through tombstones (they do not mean "stop"), but `put` is allowed to reuse a tombstone slot for a new key. This restores correctness, at the cost of slightly more bookkeeping and the fact that tombstones themselves count toward the load factor — a table with many deletions needs periodic rebuilding (rehashing into a fresh array) to actually reclaim that space, not just resizing on growth.

In [ ]:
class TombstoneHashTable:
    _EMPTY = None
    _TOMBSTONE = object()  # a unique sentinel, distinct from None and from any real key

    def __init__(self, size: int = 10):
        self.size = size
        self.slots = [self._EMPTY] * size

    def _hash(self, key) -> int:
        return sum(ord(ch) for ch in str(key)) % self.size

    def put(self, key, value) -> None:
        index = self._hash(key)
        first_tombstone = None
        for _ in range(self.size):
            slot = self.slots[index]
            if slot is self._EMPTY:
                # prefer reusing an earlier tombstone, if we passed one, to reclaim space
                target = first_tombstone if first_tombstone is not None else index
                self.slots[target] = (key, value)
                return
            if slot is self._TOMBSTONE:
                if first_tombstone is None:
                    first_tombstone = index
            elif slot[0] == key:
                self.slots[index] = (key, value)  # update existing key in place
                return
            index = (index + 1) % self.size
        raise RuntimeError("table full")

    def get(self, key):
        index = self._hash(key)
        for _ in range(self.size):
            slot = self.slots[index]
            if slot is self._EMPTY:
                break  # a true empty slot means the key was never inserted this far
            if slot is not self._TOMBSTONE and slot[0] == key:
                return slot[1]
            index = (index + 1) % self.size  # keep probing PAST tombstones -- do not stop here
        raise KeyError(key)

    def delete(self, key) -> None:
        index = self._hash(key)
        for _ in range(self.size):
            slot = self.slots[index]
            if slot is self._EMPTY:
                raise KeyError(key)
            if slot is not self._TOMBSTONE and slot[0] == key:
                self.slots[index] = self._TOMBSTONE  # leave a marker, do NOT reset to None
                return
            index = (index + 1) % self.size
        raise KeyError(key)


# Reproduce the failure mode described above, and show the tombstone fixes it
t = TombstoneHashTable(size=5)
# Force a 3-way collision chain by using keys we know share a home index under this hash
# (demonstrated generically here by inserting enough keys that some collision is forced)
t.put("A", 1)
t.put("B", 2)
t.put("C", 3)
t.put("D", 4)
assert t.get("A") == 1 and t.get("B") == 2 and t.get("C") == 3 and t.get("D") == 4

t.delete("B")   # deletes a possibly-mid-chain entry
# every OTHER key must still be reachable, even though a slot in the middle of a
# probe sequence is now a tombstone rather than truly empty
assert t.get("A") == 1
assert t.get("C") == 3
assert t.get("D") == 4
try:
    t.get("B")
    raise AssertionError("B should have been deleted")
except KeyError:
    pass

print("Tombstone-based deletion preserves reachability of later-inserted colliding keys")


**Discussion point for MTech:** the tombstone technique is exactly why Python's `dict` documentation warns that mutating a dict's size during iteration is unsafe, and why long-lived dicts with heavy churn (many inserts and deletes) can grow their internal table larger than the live key count would suggest — tombstones (CPython calls its dummy-slot equivalent similarly) occupy space until a resize forces a full rehash. The same idea appears again, later in your studies, in open-addressing variants used by many production key-value stores: correctness under deletion is a genuinely separate design problem from correctness under insertion, and it is easy to get right for insert-only workloads while still shipping a deletion bug.